# Social Media Topic Modeling Pipeline — BERTopic + OpenAI

A practical, end-to-end pipeline for clustering a large corpus of social media comments into clean, human-readable topics.

**Pipeline architecture**

```
Raw comments
   -> light cleaning (urls/mentions/whitespace)
   -> GPU sentence embeddings (SentenceTransformer)
   -> UMAP (dimensionality reduction)
   -> HDBSCAN (clustering)
   -> c-TF-IDF (initial keywords per topic)
   -> KeyBERTInspired + MaximalMarginalRelevance (cleaner keywords)
   -> OpenAI GPT (turns keywords + sample comments into one readable label)
   -> Outlier reduction (rescues comments dumped into "no topic")
   -> Final labeled dataframe + saved model
```

This follows BERTopic's own "best practices" guide (precomputed embeddings, tuned UMAP/HDBSCAN,
chained representation models, outlier reduction) plus an LLM labeling step on top, since raw
keyword sets ("lol, ngl, fr, vibe") aren't very readable for social media text.

> Run each cell top to bottom. Cells that take a while (embeddings, fitting) are marked.


## 0. Setup
Install once, then comment this cell out.

In [1]:
# !pip install -q "bertopic>=0.16,<0.17" sentence-transformers openai tiktoken \
#     umap-learn hdbscan scikit-learn pandas numpy plotly python-dotenv
#
# IMPORTANT (GPU): if `torch.cuda.is_available()` is False below, your pip install gave you
# the CPU-only build of torch. Install the CUDA build first from https://pytorch.org/get-started/locally/
# e.g.: !pip install torch --index-url https://download.pytorch.org/whl/cu121


In [2]:
import os
import re
import json
import numpy as np
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, OpenAI as BERTopicOpenAI

import openai
import tiktoken

c:\Users\visha\anaconda3\envs\bertopic\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Config
Keep secrets and knobs in one place.

In [3]:
# --- Paths ---
INPUT_CSV   = "../data/social_media_comments.csv"   # must contain a text column, see below
TEXT_COL    = "comment"                     # rename to whatever your column is called
TIMESTAMP_COL = "created_at"                # optional, set to None if you don't have one
OUTPUT_DIR  = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- OpenAI ---
# Set this in your shell instead of hardcoding it: export OPENAI_API_KEY="sk-..."
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
print(OPENAI_API_KEY[-10:])
OPENAI_LABEL_MODEL = "gpt-4o-mini"   # cheap + good enough for short topic labels

# --- Embedding model (runs on your GPU) ---
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"   # fast + solid for short, informal text
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Embedding device:", DEVICE)

assert OPENAI_API_KEY, "Set OPENAI_API_KEY as an environment variable before continuing."


5qwUCZcOgA
Embedding device: cuda


## 2. Load your data

In [4]:
df = pd.read_csv(INPUT_CSV)
print(df.shape)
df.head()


(500, 4)


,comment,platform,created_at,topic_category
0,Vinyl collection is getting out of control lol,instagram,2024-01-24 12:22:03,music
1,Humidity today is genuinely unbearable outside,facebook,2024-04-20 23:22:04,weather_climate
2,Rewatching the original trilogy and it still h...,facebook,2024-07-24 11:42:11,movies_tv
3,"can't believe he missed that penalty, easy goa...",twitter,2024-08-29 07:16:05,sports
4,Lyrics on this one are surprisingly deep 😭,instagram,2024-04-02 23:51:26,music


## 3. Light cleaning

BERT-style embeddings already understand context, slang, and informal grammar, so we deliberately
do **minimal** cleaning: strip URLs/mentions/extra whitespace, drop empties/dupes/too-short comments.
Over-cleaning (stemming, removing stopwords, lowercasing aggressively) tends to *hurt* embedding quality.


In [5]:
URL_RE = re.compile(r"http\S+|www\.\S+")
MENTION_RE = re.compile(r"@\w+")
WHITESPACE_RE = re.compile(r"\s+")

def clean_text(text: str) -> str:
    text = str(text)
    text = URL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = WHITESPACE_RE.sub(" ", text).strip()
    return text

df["clean_text"] = df[TEXT_COL].astype(str).map(clean_text)

before = len(df)
df = df[df["clean_text"].str.len() >= 8]          # drop near-empty comments
df = df.drop_duplicates(subset="clean_text")       # drop exact duplicate spam/bot comments
df = df.reset_index(drop=True)
print(f"Dropped {before - len(df)} rows (too short / duplicate). Remaining: {len(df)}")

docs = df["clean_text"].tolist()


Dropped 173 rows (too short / duplicate). Remaining: 327


## 4. Embeddings (GPU)

We pre-compute embeddings once and hand them to BERTopic directly. This is the single biggest
speed-up for large corpora, and it also lets you re-run BERTopic with different UMAP/HDBSCAN
settings without re-embedding every time.


In [6]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE)

embeddings_path = os.path.join(OUTPUT_DIR, "embeddings.npy")

if os.path.exists(embeddings_path):
    embeddings = np.load(embeddings_path)
    print("Loaded cached embeddings:", embeddings.shape)
else:
    embeddings = embedding_model.encode(
        docs,
        batch_size=128,
        show_progress_bar=True,
        device=DEVICE,
    )
    np.save(embeddings_path, embeddings)
    print("Computed embeddings:", embeddings.shape)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6418.64it/s]


Loaded cached embeddings: (327, 384)


In [7]:
embeddings

array([[ 0.02942469,  0.02283746,  0.02330599, ..., -0.1257357 ,
        -0.07159414,  0.08818418],
       [ 0.02048788,  0.07473396,  0.15273903, ..., -0.01458517,
        -0.11367095,  0.06296515],
       [-0.04782931, -0.0953863 ,  0.03381634, ..., -0.1223032 ,
        -0.02579093,  0.04717396],
       ...,
       [ 0.04175542, -0.02654319,  0.03454313, ..., -0.03977088,
        -0.03694729, -0.03624389],
       [-0.03346436, -0.01655206,  0.0378554 , ..., -0.08299068,
        -0.02666427, -0.0172061 ],
       [ 0.07677115,  0.00687077, -0.00421444, ..., -0.02719728,
         0.01780659, -0.01755358]], shape=(327, 384), dtype=float32)

## 5. Dimensionality reduction — UMAP

In [8]:
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42,   # fixes UMAP's randomness so results are reproducible
)


## 6. Clustering — HDBSCAN

`min_cluster_size` is the most important knob here: bigger = fewer, broader topics;
smaller = more, narrower topics (and more noise/outliers). Start around 30-50 for
a large social media corpus and adjust after looking at the results.


In [9]:
hdbscan_model = HDBSCAN(
    min_cluster_size=8,
    min_samples=3,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,   # required if you want .approximate_distribution() later
)


## 7. Vectorizer + c-TF-IDF tuning

`CountVectorizer` controls which words/phrases are even eligible to describe a topic.
`ClassTfidfTransformer` with `reduce_frequent_words` and `bm25_weighting` is a BERTopic
"best practice" that stops generic words (e.g. "people", "thing", "just") from dominating
every topic.


In [10]:
vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=1,           # ignore rare typos / one-off words
    max_df=1,          # ignore words that show up in almost every topic
)

ctfidf_model = ClassTfidfTransformer(
    reduce_frequent_words=True,
    bm25_weighting=True,
)


## 8. Representation models — keywords -> one readable label

These are **chained**: c-TF-IDF produces raw keywords, `KeyBERTInspired` re-ranks them by how
well they actually represent the topic semantically, `MaximalMarginalRelevance` removes near-duplicate
keywords, and finally the OpenAI model reads the cleaned keywords + a handful of real comments and
writes one short, human label. Only the OpenAI step uses your API key/credits, and it's cheap
because it only sees a few keywords + a few example comments per topic — not your whole corpus.


In [11]:
client = openai.OpenAI(api_key=OPENAI_API_KEY)
tokenizer = tiktoken.get_encoding("cl100k_base")

label_prompt = """
I have a topic from a large set of social media comments.
The topic is described by these keywords: [KEYWORDS]
Here are a few example comments from this topic:
[DOCUMENTS]

Write a short, specific topic label (3-6 words, no quotes, no punctuation at the end)
that a human moderator could use to understand what this group of comments is about.
Respond with only the label.
"""

openai_label_model = BERTopicOpenAI(
    client,
    model=OPENAI_LABEL_MODEL,
    chat=True,
    prompt=label_prompt,
    nr_docs=6,            # how many example comments to show the LLM per topic
    doc_length=120,        # truncate each comment to this many tokens
    tokenizer=tokenizer,
    delay_in_seconds=1,    # gentle rate-limiting; raise if you hit rate limits
    exponential_backoff=True,
)

# Chained: c-TF-IDF -> KeyBERTInspired -> MMR -> OpenAI label
representation_model = [
    KeyBERTInspired(),
    MaximalMarginalRelevance(diversity=0.3),
    openai_label_model,
]


## 9. Assemble & fit BERTopic

In [12]:
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=representation_model,
    top_n_words=10,
    calculate_probabilities=False,   # set True only if you need full soft-probability matrices
    verbose=True,
)

topics, _ = topic_model.fit_transform(docs, embeddings)
print("Number of topics found (excluding outliers):", len(topic_model.get_topic_info()) - 1)


2026-06-27 14:11:47,883 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-27 14:12:04,032 - BERTopic - Dimensionality - Completed ✓
2026-06-27 14:12:04,034 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-27 14:12:04,044 - BERTopic - Cluster - Completed ✓
2026-06-27 14:12:04,049 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 18/18 [00:31<00:00,  1.77s/it]
2026-06-27 14:12:37,451 - BERTopic - Representation - Completed ✓


Number of topics found (excluding outliers): 17


## 10. Reduce outliers

HDBSCAN labels anything that doesn't fit cleanly into a cluster as topic `-1` ("outlier").
Social media text is noisy, so this bucket can be large. `reduce_outliers` reassigns those
comments to their nearest real topic using embedding similarity, instead of throwing them away.


In [13]:
new_topics = topic_model.reduce_outliers(docs, topics, strategy="embeddings", embeddings=embeddings)

topic_model.update_topics(
    docs,
    topics=new_topics,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=representation_model,
)

topics = new_topics


2026-06-27 14:12:37,528 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


100%|██████████| 17/17 [00:29<00:00,  1.72s/it]


## 11. Inspect the topics

In [14]:
topic_info = topic_model.get_topic_info()
topic_info.head(20)


,Topic,Count,Name,Representation,Representative_Docs
0,0,67,0_Defeating the Final Boss,[Defeating the Final Boss],[new collection drop sold out within minutes a...
1,1,30,1_Router firmware update issues,[Router firmware update issues],[Anyone else having wifi issues since the late...
2,2,23,2_Budget Proposal and Institutional Buy,[Budget Proposal and Institutional Buy],[Budget proposal includes some surprising cuts...
3,3,23,3_Coaching changes and player injuries,[Coaching changes and player injuries],[Injury report not looking good for our star p...
4,4,21,4_Luggage and Package Damage Issues,[Luggage and Package Damage Issues],[Lost my luggage but the airline handled it pr...
5,5,20,5_Affordable New Accessory Line,[Affordable New Accessory Line],[New accessory line looks surprisingly afforda...
6,6,15,6_Online Debates and Housing Issues,[Online Debates and Housing Issues],[honestly policy debate online got way too hea...
7,7,15,7_Deadlift Achievements and Shoulder Issues,[Deadlift Achievements and Shoulder Issues],[hit a new deadlift pr today feeling unstoppab...
8,8,17,8_Cats Knocking Things Over,[Cats Knocking Things Over],[Cat knocked everything off my desk again this...
9,9,13,9_Hurricane and Weather Concerns,[Hurricane and Weather Concerns],[honestly flooding in the area is way worse th...


In [15]:
print(topic_model.get_topic_info())

    Topic  Count                                         Name  \
0       0     67                   0_Defeating the Final Boss   
1       1     30              1_Router firmware update issues   
2       2     23      2_Budget Proposal and Institutional Buy   
3       3     23       3_Coaching changes and player injuries   
4       4     21          4_Luggage and Package Damage Issues   
5       5     20              5_Affordable New Accessory Line   
6       6     15          6_Online Debates and Housing Issues   
7       7     15  7_Deadlift Achievements and Shoulder Issues   
8       8     17                  8_Cats Knocking Things Over   
9       9     13             9_Hurricane and Weather Concerns   
10     10     14          10_Protein shake and baking recipes   
11     11     15         11_Therapy and Self-Care Reflections   
12     12     15              12_Celebrating Small Daily Wins   
13     13     11      13_Crowd and Voter Turnout Expectations   
14     14     10         

In [16]:
# Interactive plots (open in browser / render inline in Jupyter)
fig_topics = topic_model.visualize_topics()
fig_topics.show()

fig_bar = topic_model.visualize_barchart(top_n_topics=12)
fig_bar.show()

## 12. Build the final labeled dataframe

In [17]:
doc_info = topic_model.get_document_info(docs)

# The OpenAI-generated label lives in the last representation model's output.
# get_topic_info()["Name"] already reflects the final (OpenAI) representation by default.
topic_id_to_label = dict(zip(topic_info["Topic"], topic_info["Name"]))

df["topic_id"] = topics
df["topic_label"] = df["topic_id"].map(topic_id_to_label)

results_path = os.path.join(OUTPUT_DIR, "labeled_comments.csv")
df.to_csv(results_path, index=False)
print("Saved:", results_path)
df[[TEXT_COL, "topic_id", "topic_label"]].head(10)


Saved: outputs\labeled_comments.csv


,comment,topic_id,topic_label
0,Vinyl collection is getting out of control lol,0,0_Defeating the Final Boss
1,Humidity today is genuinely unbearable outside,9,9_Hurricane and Weather Concerns
2,Rewatching the original trilogy and it still h...,0,0_Defeating the Final Boss
3,"can't believe he missed that penalty, easy goa...",3,3_Coaching changes and player injuries
4,Lyrics on this one are surprisingly deep 😭,0,0_Defeating the Final Boss
5,That live performance gave me actual chills,0,0_Defeating the Final Boss
6,That dessert place raised their prices way too...,15,15_Cloud Storage Subscription Price Hikes
7,Opening act completely stole the show tonight,0,0_Defeating the Final Boss
8,New exchange listing caused a small price pump 🔥,2,2_Budget Proposal and Institutional Buy
9,honestly new jersey design is actually fire ngl,14,14_Balance and Nerf Discussions


## 13. Save the model for reuse

In [18]:
model_path = os.path.join(OUTPUT_DIR, "bertopic_model")
topic_model.save(
    model_path,
    serialization="safetensors",
    save_ctfidf=True,
    save_embedding_model=EMBEDDING_MODEL_NAME,
)
print("Model saved to:", model_path)

# To reload later:
# topic_model = BERTopic.load(model_path)


Model saved to: outputs\bertopic_model


## 14. Optional advanced extensions

Use these if your data supports them — none are required for the core pipeline above.


In [19]:
# A) Topics over time (needs a real timestamp column)
if TIMESTAMP_COL and TIMESTAMP_COL in df.columns:
    timestamps = pd.to_datetime(df[TIMESTAMP_COL]).tolist()
    topics_over_time = topic_model.topics_over_time(docs, timestamps, nr_bins=20)
    topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=10).show()
else:
    print("Skipping topics-over-time: no timestamp column configured.")


100%|██████████| 9/9 [00:17<00:00,  1.92s/it]
20it [05:48, 17.40s/it]


In [20]:
# B) Topics per category, e.g. per platform/subreddit/source (needs a class column)
CLASS_COL = "platform"  # change to your column name, or set to None to skip
if CLASS_COL and CLASS_COL in df.columns:
    topics_per_class = topic_model.topics_per_class(docs, classes=df[CLASS_COL].tolist())
    topic_model.visualize_topics_per_class(topics_per_class, top_n_topics=10).show()
else:
    print("Skipping topics-per-class: no class column configured.")


100%|██████████| 17/17 [00:31<00:00,  1.85s/it]
5it [02:33, 30.72s/it]


In [21]:
# C) Soft topic membership instead of one hard topic per comment
# Useful when a comment plausibly belongs to more than one topic.
# Requires hdbscan_model created with prediction_data=True (already done above).
topic_distr, _ = topic_model.approximate_distribution(docs[:1000])  # demo on a subset
print(topic_distr.shape)  # (n_docs, n_topics) — each row sums to ~1


100%|██████████| 1/1 [00:00<00:00, 36.28it/s]

(327, 17)


### Notes on scaling to a *very* large corpus

- Embeddings are the expensive part: encode in batches on GPU (already done above), and cache
  the `.npy` file so you never recompute them for the same comments.
- If the corpus is too large to fit in memory at once, use BERTopic's `.partial_fit()` /
  `merge_models()` to train on chunks and merge the resulting topic models.
- `UMAP` memory grows roughly quadratically with `n_neighbors` and dataset size — lower
  `n_neighbors` or sample down for an initial exploratory pass before committing to a full run.
- The OpenAI labeling step only runs once per *topic*, not once per comment, so cost stays
  small even with millions of comments — you'll typically have far fewer than a few hundred topics.
